# Revisiting FITS

**Motivating question:** Oh no, there are negative TARGETIDs. Did I cast dtypes incorrectly?  
**Answer (at bottom):** It's ok; the negtives should corresond with non-target OBJTYPEs!

In [3]:
import sys
from pathlib import Path

import numpy as np


np.set_printoptions(threshold=sys.maxsize)

In [4]:
# Existing catalogs

main_directory = Path("/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron")
input_path = main_directory / "healpix/"

# Select a test file to inspect: healpix group 0, healpix 0

test_file = input_path / "main/dark/0/0/coadd-main-dark-0.fits"

## Confirming the FITS seem to have negative TARGETIDs

If this happens before I even touch the data...hmm...

### Using astropy

In [46]:
# See the first 10 entries of the TARGETID column in the FIBERMAP extension of the FITS file

from astropy.io import fits


with fits.open(test_file) as hdul:
    fibermap_cols = hdul["FIBERMAP"].columns
    print(fibermap_cols["TARGETID"])  # K : 64-bit integer

    table_col = hdul["FIBERMAP"].data["TARGETID"][:10]

    print(f"TARGETID col type: {table_col.dtype}\n")  # >i8 : big-endian 64-bit integer

    print(table_col)


name = 'TARGETID'; format = 'K'
TARGETID col type: >i8

[39627791519454588 39627791519452760 39627791519456177 39627785479655815
 39627791519455650 39627791519454413 39627791519452686         -48140048
 39627785483846233 39627791519456493]


### Using fits.io

In [17]:
import fitsio
from fitsio import FITS,FITSHDR

In [35]:
with fitsio.FITS(test_file) as fits:
    print(fits["FIBERMAP"]["TARGETID"], "\n")
    print(fits["FIBERMAP"].read(rows=range(10), columns=["TARGETID"]))

  file: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/coadd-main-dark-0.fits
  extension: 1
  type: BINARY_TBL
  rows: 1704
  column subset:
    TARGETID            i8   

[(39627791519454588,) (39627791519452760,) (39627791519456177,)
 (39627785479655815,) (39627791519455650,) (39627791519454413,)
 (39627791519452686,) (        -48140048,) (39627785483846233,)
 (39627791519456493,)]


## Comparing with OBJTYPE

Chatted with a really helpful guy a few doors down. Apparently, this is a-ok.

In [ ]:
# Notes from quick chat down the hall:
# - neg numbers are skyfibers - like, background
# - check out OBJTYPE column
# - also, if the TARGETID starts with a 2, that means something
#   - but what :) he could not recall, tho i still appreciate the tip
# - in DESI they often keep ids as strings, he said
#   - maybe I should plan to do this as well?
#   - though--if the DESI datamodel site lists TARGETID as a 64bit int, seems find to preserve that...

In [5]:
import pandas as pd
from astropy.io import fits

with fits.open(test_file) as hdul:
    fibermap_data = hdul["FIBERMAP"].data
    
    df = pd.DataFrame({
        'TARGETID': fibermap_data["TARGETID"][:10],
        'OBJTYPE': fibermap_data["OBJTYPE"][:10],
        'TARGET_RA': fibermap_data["TARGET_RA"][:10],
        'TARGET_DEC': fibermap_data["TARGET_DEC"][:10],
    })
    print(df)

            TARGETID OBJTYPE  TARGET_RA  TARGET_DEC
0  39627791519454588     TGT  44.925441    0.149561
1  39627791519452760     TGT  44.872364    0.256157
2  39627791519456177     TGT  44.979791    0.158020
3  39627785479655815     TGT  44.916675    0.094066
4  39627791519455650     TGT  44.961076    0.272126
5  39627791519454413     TGT  44.919962    0.215757
6  39627791519452686     TGT  44.870136    0.186088
7          -48140048     SKY  44.931005    0.120554
8  39627785483846233     TGT  45.020772    0.115927
9  39627791519456493     TGT  44.990819    0.156522


In [ ]:
# Action items:
# - Store TARRGETID as string, maybe? - tho, maybe not...
# - Add OBJTYPE col
# - Add SPECTYPE col, too, while we're at it